# Read in Python Libraries:

In [5]:
import warnings
warnings.filterwarnings("ignore")


import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
import matplotlib.pyplot as plt
import os

import seaborn as sns

from sklearn.metrics import r2_score

from scipy.stats import genextreme


import glob
import openpyxl

import altair as alt
import os


# Read in data

In [6]:
import os
import re
import pandas as pd
import numpy as np

# Function to convert number to Excel column letter
def get_column_letter(n):
    result = ""
    while n > 0:
        n, remainder = divmod(n - 1, 26)
        result = chr(65 + remainder) + result
    return result

# File to process
file_path = 'CORE_DATA_AND_DESCRIPTIONS_DATABASE_HPMI.xlsx'

# Output directory
output_dir = './Cleaned'
os.makedirs(output_dir, exist_ok=True)

# Sheet number (adjust if needed)
sheet_number = 0  

# Read Pc values (same for all samples)
pc_values = pd.read_excel(file_path, sheet_name=sheet_number, usecols="A", skiprows=11, nrows=119)
pc_values = pc_values.values.flatten()

# Output cleaned file path
cleaned_file_path = os.path.join(output_dir, "cleaned_CORE_DATA_AND_DESCRIPTIONS_DATABASE_HPMI.xlsx")

with pd.ExcelWriter(cleaned_file_path, engine='xlsxwriter') as writer:
    # Loop through samples in columns B to desired last column
    for sample_num in range(1, 36):  # adjust 36 if more samples
        column_letter = get_column_letter(sample_num + 1)  # +1 since 'B' is column 2

        # Read Sw values (% in file → fraction here)
        sw_values = pd.read_excel(file_path, sheet_name=sheet_number, usecols=column_letter, skiprows=11, nrows=119)
        sw_values = sw_values.values.flatten() / 100.0

        # Metadata
        porosity = pd.read_excel(file_path, sheet_name=sheet_number, usecols=column_letter, skiprows=7, nrows=1).iloc[0, 0]
        permeability = pd.read_excel(file_path, sheet_name=sheet_number, usecols=column_letter, skiprows=8, nrows=1).iloc[0, 0]

        # Derived
        SHg = 1 - sw_values
        BVocc = SHg * porosity  # porosity is in %, so BVocc is % of bulk volume

        # Flatten format (metadata in first row only)
        rows = []
        for j in range(len(pc_values)):
            if j == 0:
                rows.append([
                    f"Sample_{sample_num}",
                    porosity,
                    permeability,
                    pc_values[j],
                    BVocc[j],
                    BVocc[j],  # BVocc_original same here
                    SHg[j]
                ])
            else:
                rows.append([
                    '', '', '',
                    pc_values[j],
                    BVocc[j],
                    BVocc[j],
                    SHg[j]
                ])

        flat_df = pd.DataFrame(rows, columns=[
            'Sample', 'Porosity', 'Permeability',
            'Pc', 'BVocc_original', 'BVocc', 'SHg'
        ])

        # Clean sheet name for Excel
        sheet_output = re.sub(r'[\\/*?:\[\]]', '_', f"Sample_{sample_num}")[:31]

        # Save each sample as its own sheet
        flat_df.to_excel(writer, sheet_name=sheet_output, index=False)

print(f"✅ Cleaned Excel file written to: {cleaned_file_path}")


✅ Cleaned Excel file written to: ./Cleaned/cleaned_CORE_DATA_AND_DESCRIPTIONS_DATABASE_HPMI.xlsx


In [ ]:
break

In [ ]:
import os
import re
import pandas as pd
import numpy as np

# Function to convert number to Excel column letter
def get_column_letter(n):
    result = ""
    while n > 0:
        n, remainder = divmod(n - 1, 26)
        result = chr(65 + remainder) + result
    return result

# File to process
file_path = 'CORE_DATA_AND_DESCRIPTIONS_DATABASE_HPMI.xlsx'

# Output directory
output_dir = './Cleaned'
os.makedirs(output_dir, exist_ok=True)

# Sheet number (adjust if needed)
sheet_number = 0  

# Read Pc values (same for all samples)
pc_values = pd.read_excel(file_path, sheet_name=sheet_number, usecols="A", skiprows=11, nrows=119)
pc_values = pc_values.values.flatten()

# Initialize ExcelWriter for cleaned output
cleaned_file_path = os.path.join(output_dir, "cleaned_CORE_DATA_AND_DESCRIPTIONS_DATABASE_HPMI.xlsx")

with pd.ExcelWriter(cleaned_file_path, engine='xlsxwriter') as writer:
    # Loop through samples in columns B to whatever last sample column is
    for sample_num in range(1, 36):  # adjust 36 if more samples
        column_letter = get_column_letter(sample_num + 1)  # +1 since 'B' is column 2

        # Read Sw values (% in file → fraction here)
        sw_values = pd.read_excel(file_path, sheet_name=sheet_number, usecols=column_letter, skiprows=11, nrows=119)
        sw_values = sw_values.values.flatten() / 100.0

        # Metadata
        porosity = pd.read_excel(file_path, sheet_name=sheet_number, usecols=column_letter, skiprows=7, nrows=1).iloc[0, 0]
        permeability = pd.read_excel(file_path, sheet_name=sheet_number, usecols=column_letter, skiprows=8, nrows=1).iloc[0, 0]
        num_pore_sys = pd.read_excel(file_path, sheet_name=sheet_number, usecols=column_letter, skiprows=9, nrows=1).iloc[0, 0]

        # Derived
        SHg = 1 - sw_values
        BVocc = SHg * porosity  # porosity is in %, so BVocc is % of bulk volume

        # Flatten format (metadata in first row only)
        rows = []
        for j in range(len(pc_values)):
            if j == 0:
                rows.append([
                    f"Sample_{sample_num}",
                    porosity,
                    permeability,
                    #num_pore_sys,
                    pc_values[j],
                    BVocc[j],
                    BVocc[j],  # BVocc_original same here
                    SHg[j]
                ])
            else:
                rows.append([
                    '', '', '', '',
                    pc_values[j],
                    BVocc[j],
                    BVocc[j],
                    SHg[j]
                ])

        flat_df = pd.DataFrame(rows, columns=[
            'Sample', 'Porosity', 'Permeability', 'Num_pore_sys',
            'Pc', 'BVocc_original', 'BVocc', 'SHg'
        ])

        # Clean sheet name for Excel
        sheet_output = re.sub(r'[\\/*?:\[\]]', '_', f"Sample_{sample_num}")[:31]

        # Save each sample in its own sheet
        flat_df.to_excel(writer, sheet_name=sheet_output, index=False)

print(f"✅ Cleaned Excel file written to: {cleaned_file_path}")


In [ ]:
break

# Read in Cleaned Cutbow and Theo Data:

## **Use Perm_BC as the actual perm to avoid fractured core plug samples.**

In [ ]:
# Interactive Altair Plots: Facies-colored Crossplot and Capillary Pressure Curves


# Load cleaned Excel data
cleaned_file = "Cleaned_HGCP_Data.xlsx"
xls = pd.ExcelFile(cleaned_file)

# Consolidate all data into a single DataFrame
all_data = []
for sheet in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet)
    df["Sample"] = sheet
    all_data.append(df)

combined_df = pd.concat(all_data, ignore_index=True)

# Drop rows with missing essential values
combined_df = combined_df.dropna(subset=["Porosity", "Perm", "Facies", "Pc_Hg", "Shg","Depth_log"])





# Add a source column before combining
combined_df["Source"] = "Cuttbow"


print("Loaded Cuttbow samples:", combined_df["Sample"].nunique())
print("Final shape:", combined_df.shape)



